# OU Path Streaming Demo

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Stream an Ornstein-Uhlenbeck path in real time with browser-side simulation.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/ou-path-stream/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(2026)

# The demo's defaults: dX = theta (mu - X) dt + sigma dW, Euler-Maruyama
T, dt, theta, mu, sigma, x0 = 10.0, 0.01, 1.2, 0.0, 0.8, 0.0
steps = round(T / dt)

# ── One long path, as streamed in the demo ──────────────────────────────────
x, path = x0, []
for z in torch.randn(200 * steps, dtype=torch.float64).tolist():      # run 200x longer for statistics
    x = x + theta * (mu - x) * dt + sigma * math.sqrt(dt) * z
    path.append(x)
path = torch.tensor(path, dtype=torch.float64)[steps:]               # drop the first T as warm-up

print(f"long-run mean     {path.mean().item():+.4f}   (mu = {mu})")
print(f"long-run variance {path.var().item():.4f}   (sigma^2 / (2 theta) = {sigma ** 2 / (2 * theta):.4f})")

# ── Mean reversion: correlation decays like exp(-theta * lag) ───────────────
c = path - path.mean()
print("\n  lag    autocorrelation   exp(-theta lag)")
for lag in (0.1, 0.5, 1.0, 2.0):
    l = round(lag / dt)
    print(f"{lag:5.1f}       {((c[:-l] * c[l:]).mean() / c.var()).item():.4f}          {math.exp(-theta * lag):.4f}")
print(f"\nrelaxation time 1/theta = {1 / theta:.3f}")

long-run mean     -0.0209   (mu = 0.0)
long-run variance 0.2662   (sigma^2 / (2 theta) = 0.2667)

  lag    autocorrelation   exp(-theta lag)
  0.1       0.8857          0.8869
  0.5       0.5473          0.5488
  1.0       0.2883          0.3012
  2.0       0.0760          0.0907

relaxation time 1/theta = 0.833
